<a href="https://colab.research.google.com/github/Riz2693/Dicoding-Submission-FDL/blob/main/Analisis%20Sentimen/scraping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Muhammad Faris Akbar**
<br></br>
**Fundamental Deep Learning - Sentimen Analisis Shopee**

In [71]:
!pip install google-play-scraper transformers

In [72]:
import sys
import os
import pandas as pd
import re

In [73]:
# try:
#     # Jika dijalankan sebagai file .py
#     base_path = os.path.dirname(os.path.abspath(__file__))
# except NameError:
#     # Jika dijalankan di Jupyter/Interactive
#     base_path = os.getcwd()

# parent_dir = os.path.abspath(os.path.join(base_path, '..'))
# sys.path.append(parent_dir)

# print("Base Path:", base_path)
# print("Parent Directory:", parent_dir)

# from Helper import *

In [74]:
from google_play_scraper import reviews, Sort
from transformers import pipeline
from tqdm import tqdm

**Helper Function**

In [75]:
def cek_nan(data):
  """
  Function yang digunakan untuk memeriksa nilai nan dari data
  input   : data
  output  : nan_info (DataFrame)
  return  : nan_info (DataFrame)
  """
  nan_info = pd.DataFrame(data.isna().sum().sort_values(ascending=False), columns=['Jumlah Nilai Missing'])

  if nan_info['Jumlah Nilai Missing'].sum() == 0:
    print("Tidak ada nilai missing")
    return None
  else:
    nan_info['Persentase Nilai Missing (%)'] = ((nan_info['Jumlah Nilai Missing'] / len(data)) * 100).round(3)

    # return data dengan nilai missing, apabila ingin mereturn keseluruhan data maka ubah menjadi return nan_info
    return nan_info[nan_info['Jumlah Nilai Missing'] > 0]

In [76]:
def visualize_row_with_nan(data, method='all', columns=None):
  """
  Procedure yang digunakan untuk menvisualisasikan baris yang mengandung NaN berdasarkan metode yang dipilih
  input   : data, method, columns
  output  : baris_nan (DataFrame)
  """
  try:
    if method not in ['all', 'column', 'columns']:
      raise ValueError("Metode tidak valid. Pilih salah satu dari 'all', 'column', 'columns'.")

    if method == 'all':
      display(data[data.isna().any(axis=1)].head(5))


    elif method == 'column':
      if not isinstance(columns, str):
          raise TypeError("Untuk metode 'column', argumen 'columns' harus berupa String.")
      if columns not in data.columns:
          raise KeyError(f"Kolom '{columns}' tidak ditemukan dalam DataFrame.")
      display(data[data[columns].isna()].head(5))

    elif method == 'columns':
      if not isinstance(columns, list):
          raise TypeError("Untuk metode 'columns', argumen 'columns' harus berupa list.")
      for column in columns:
          if column not in data.columns:
              raise KeyError(f"Kolom '{column}' tidak ditemukan dalam DataFrame.")
      display(data[data[columns].isna().all(axis=1)].head(5))

  except (TypeError, KeyError) as e:
    print(f"Kesalahan dalam memproses data: {e}")

In [77]:
def visualize_row_with_duplicated(data):
  """
  Procedure yang digunakan untuk menvisualisasikan baris yang mengandung data duplikat
  input   : data
  output  : baris_duplikat (DataFrame)
  """
  duplicated = data.duplicated().sum()

  if duplicated > 0:
    print("Jumlah Data Duplikat :", duplicated)
    all_duplicates = data[data.duplicated(keep=False)]

    duplicate_indices = all_duplicates.groupby(list(all_duplicates.columns)).groups

    print("Contoh 10 Pasangan Data Duplikat :")
    for group_indices in list(duplicate_indices.values())[:10]:
      if len(group_indices) > 1:
        display(data.iloc[group_indices])
        print("\n")
  else:
    print("Tidak ada data duplikat")

**Main Section**

In [78]:
def scrape_google_play(app_id, total_count, country='id', lang='id'):
    """
    Fungsi untuk melakukan scraping ulasan dari Google Play Store.

    app_id (str): ID aplikasi di Play Store (contoh: 'com.shopee.id')
    total_count (int): Target jumlah data yang ingin diambil
    country (str): Kode negara (default Indonesia 'id')
    lang (str): Bahasa ulasan (default Indonesia 'id')
    """
    print(f"Proses scraping untuk aplikasi: {app_id}...")

    result, continuation_token = reviews(
        app_id,
        lang=lang,
        country=country,
        sort=Sort.NEWEST, # Sort.NEWEST digunakan untuk mendapat data terbaru
        count=total_count,
        filter_score_with=None # Mengambil semua rating (1-5)
    )

    print(f"Berhasil mengambil {len(result)} data mentah.")

    # Konversi hasil scraping ke dalam DataFrame (Tabel)
    df = pd.DataFrame(result)

    return df

In [79]:
TARGET_APP = 'com.shopee.id'
JUMLAH_DATA = 100000

df_ulasan = scrape_google_play(TARGET_APP, JUMLAH_DATA)

Proses scraping untuk aplikasi: com.shopee.id...
Berhasil mengambil 100000 data mentah.


In [80]:
# df_ulasan = pd.read_csv('https://raw.githubusercontent.com/ArmFriiz/Dicoding-Submission-FDL/refs/heads/main/Analisis%20Sentimen/dataset_ulasan_playstore.csv')

In [81]:
df_ulasan.head(5)

,reviewId,userName,userImage,content,score,thumbsUpCount,reviewCreatedVersion,at,replyContent,repliedAt,appVersion
0,b68f7f71-a359-4f34-b514-a4fd98b68498,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,"suka belanja di shopee,jaya selalu shopee",5,0,3.68.37,2026-03-06 12:10:20,"Hi Kak Isah Parisah, Terimakasih ya buat revie...",2026-03-06 13:32:25,3.68.37
1,a38802e3-b97f-49fa-ac3c-13a15381d61e,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,Banyak ongkir gratis nya gak akan rugi beli di...,5,0,None,2026-03-06 12:09:31,"Hi Kak Wesly Saragih, makasih bgt ya review da...",2026-03-06 13:36:14,None
2,63dfb413-bf99-40e6-a9c1-ba4b98b472f0,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,saya sangat puas harga nya murah murah,5,0,3.68.37,2026-03-06 12:09:10,"Hi Kak Siti aulia Silva, makasih bgt ya review...",2026-03-06 13:38:09,3.68.37
3,9cb122e2-3cf5-4601-85c8-9342529cf021,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,belanja jadi mudah dan menyenangkan tak perlu ...,5,0,3.68.42,2026-03-06 12:07:31,"Hi Kak wila mardiana, makasih bgt ya review da...",2026-03-06 13:38:36,3.68.42
4,da8095ff-ecd9-4f97-8deb-04ef552e8ae9,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,kurirnya ga jelas kirim pesan tidak menghubung...,1,0,3.68.42,2026-03-06 12:05:25,"Hai kak Dzul Fadli, maaf ya udah buat risau te...",2026-03-06 13:33:08,3.68.42


In [82]:
df_ulasan.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 11 columns):
 #   Column                Non-Null Count   Dtype         
---  ------                --------------   -----         
 0   reviewId              100000 non-null  object        
 1   userName              100000 non-null  object        
 2   userImage             100000 non-null  object        
 3   content               100000 non-null  object        
 4   score                 100000 non-null  int64         
 5   thumbsUpCount         100000 non-null  int64         
 6   reviewCreatedVersion  78070 non-null   object        
 7   at                    100000 non-null  datetime64[ns]
 8   replyContent          94367 non-null   object        
 9   repliedAt             94367 non-null   datetime64[ns]
 10  appVersion            78070 non-null   object        
dtypes: datetime64[ns](2), int64(2), object(7)
memory usage: 8.4+ MB


In [83]:
df_ulasan.describe(include='all')

,reviewId,userName,userImage,content,score,thumbsUpCount,reviewCreatedVersion,at,replyContent,repliedAt,appVersion
count,100000,100000,100000,100000,100000.000000,100000.000000,78070,100000,94367,94367,78070
unique,100000,1000,1000,74442,NaN,NaN,374,NaN,92379,NaN,374
top,104ed851-ff63-4415-9308-cd0c987b7950,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,bagus,NaN,NaN,3.63.26,NaN,"Hai Kak , makasih ya buat penilaiannya, semoga...",NaN,3.63.26
freq,1,99001,99001,3086,NaN,NaN,8480,NaN,591,NaN,8480
mean,NaN,NaN,NaN,NaN,4.014160,1.937340,NaN,2026-01-14 03:45:38.918899968,NaN,2025-12-19 10:46:07.023292160,NaN
min,NaN,NaN,NaN,NaN,1.000000,0.000000,NaN,2025-12-02 08:05:38,NaN,2017-08-12 07:51:16,NaN
25%,NaN,NaN,NaN,NaN,3.000000,0.000000,NaN,2025-12-19 09:33:44,NaN,2025-12-17 05:33:01.500000,NaN
50%,NaN,NaN,NaN,NaN,5.000000,0.000000,NaN,2026-01-10 13:06:09.500000,NaN,2026-01-08 11:36:43,NaN
75%,NaN,NaN,NaN,NaN,5.000000,0.000000,NaN,2026-02-07 08:53:56.750000128,NaN,2026-02-05 21:25:39,NaN
max,NaN,NaN,NaN,NaN,5.000000,16023.000000,NaN,2026-03-06 12:10:20,NaN,2026-03-06 14:53:21,NaN


**Cek Validitas dan Kebersihan Data**

In [84]:
cek_nan(df_ulasan)

,Jumlah Nilai Missing,Persentase Nilai Missing (%)
reviewCreatedVersion,21930,21.930
appVersion,21930,21.930
replyContent,5633,5.633
repliedAt,5633,5.633


In [85]:
visualize_row_with_nan(df_ulasan)

,reviewId,userName,userImage,content,score,thumbsUpCount,reviewCreatedVersion,at,replyContent,repliedAt,appVersion
1,a38802e3-b97f-49fa-ac3c-13a15381d61e,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,Banyak ongkir gratis nya gak akan rugi beli di...,5,0,None,2026-03-06 12:09:31,"Hi Kak Wesly Saragih, makasih bgt ya review da...",2026-03-06 13:36:14,None
7,55c45dae-27d2-4d96-a799-63b04eb1d226,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,banyak biaya ini itu apalagi sekarang biaya la...,1,0,3.68.42,2026-03-06 12:03:36,None,NaT,3.68.42
13,43917d20-06d5-4bb5-a18b-f8beaf4c70ab,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,sering diretur sama kurir nya udh 3 kali jadi ...,1,0,None,2026-03-06 11:58:09,"Hi kak Adictia Pramono, mohon maaf atas ketida...",2026-03-06 13:27:10,None
17,c5808a0c-3330-48be-98d4-99d4cba4d814,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,kenapa pengembalian barang selalu di tolak pad...,1,0,None,2026-03-06 11:55:51,"Haii kak anjas rizki , maaf ya udh bikin km k...",2026-03-06 09:44:14,None
19,b4d6969e-735b-49c5-9a22-4f7b564f57b2,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,baguss,3,0,None,2026-03-06 11:53:03,"Wuiihhh makasih banyak review ,aku tambah sene...",2026-03-06 13:22:35,None


In [86]:
visualize_row_with_duplicated(df_ulasan)

Tidak ada data duplikat


**Filter kolom content dan score untuk konten data**

In [87]:
df = df_ulasan[['content', 'score']]

In [88]:
visualize_row_with_duplicated(df)

Jumlah Data Duplikat : 24816
Contoh 10 Pasangan Data Duplikat :


,content,score
21853,",👍",5
33033,",👍",5


,content,score
51435,Alhamdulilah,5
82268,Alhamdulilah,5


,content,score
671,Alhamdulillah,5
1624,Alhamdulillah,5
3856,Alhamdulillah,5
5447,Alhamdulillah,5
5483,Alhamdulillah,5
7159,Alhamdulillah,5
9087,Alhamdulillah,5
10003,Alhamdulillah,5
15309,Alhamdulillah,5
15417,Alhamdulillah,5


,content,score
54779,Alhamdulillah amanah,5
94739,Alhamdulillah amanah,5


...

,content,score
23444,Alhamdulillah barang sesuai pesanan,5
26712,Alhamdulillah barang sesuai pesanan,5


,content,score
15076,Alhamdulillah membantu,5
85677,Alhamdulillah membantu,5


,content,score
37651,Alhamdulillah puas,5
74847,Alhamdulillah puas,5


,content,score
50401,Alhamdulillah tidak pernah mengecewakan,5
71434,Alhamdulillah tidak pernah mengecewakan,5


,content,score
12147,Amanah,5
23892,Amanah,5
92872,Amanah,5


In [89]:
df.drop_duplicates(inplace=True)

/tmp/ipykernel_15799/3006716147.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.drop_duplicates(inplace=True)


In [90]:
visualize_row_with_duplicated(df)

Tidak ada data duplikat


In [91]:
print(f"Ukuran Data Setelah Pembersihan NaN dan Duplicated: {df.shape}")

Ukuran Data Setelah Pembersihan NaN dan Duplicated: (75184, 2)


**Soft Cleaning Data**

In [92]:
def cleaning_untuk_labeling(text):
    text = str(text)
    text = re.sub(r'https?://\S+|www\.\S+', '', text) # Hapus URL
    text = re.sub(r'<.*?>', '', text) # Hapus HTML tags
    text = re.sub(r'@[A-Za-z0-9_]+', '', text) # Hapus Mentions (@user)
    text = re.sub(r'#[A-Za-z0-9]+', '', text) # Hapus hashtag
    text = re.sub(r'(.)\1{2,}', r'\1\1', text) # Hapus kata berulang yang muncul lebih dari 2x
    text = re.sub(r'[a-zA-Z]+\d+\w*|\w*\d+[a-zA-Z]+', '', text) # Hapus kombinasi angka dan huruf seperti m4ndi, 4yam, dll
    text = re.sub(r'\b\d{7,}\b', '', text) # Hapus angka yang panjangnya lebih dari 7
    text = ' '.join(text.split())

    return text

In [93]:
df['soft_clean_content'] = df['content'].apply(cleaning_untuk_labeling)

/tmp/ipykernel_15799/3141896218.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['soft_clean_content'] = df['content'].apply(cleaning_untuk_labeling)


In [94]:
df.head(5)

,content,score,soft_clean_content
0,"suka belanja di shopee,jaya selalu shopee",5,"suka belanja di shopee,jaya selalu shopee"
1,Banyak ongkir gratis nya gak akan rugi beli di...,5,Banyak ongkir gratis nya gak akan rugi beli di...
2,saya sangat puas harga nya murah murah,5,saya sangat puas harga nya murah murah
3,belanja jadi mudah dan menyenangkan tak perlu ...,5,belanja jadi mudah dan menyenangkan tak perlu ...
4,kurirnya ga jelas kirim pesan tidak menghubung...,1,kurirnya ga jelas kirim pesan tidak menghubung...


In [95]:
cek_nan(df)

Tidak ada nilai missing


In [96]:
visualize_row_with_nan(df)

,content,score,soft_clean_content


In [97]:
df.dropna(inplace=True)

/tmp/ipykernel_15799/1379821321.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.dropna(inplace=True)


**Labeling Data**

In [98]:
# def labeling_data(df):
#     """
#     Fungsi untuk memberikan label otomatis berdasarkan skor bintang.
#     Menggunakan logika:
#     1-2 Bintang = Negatif
#     3 Bintang   = Netral
#     4-5 Bintang = Positif
#     """
#     def get_sentiment(score):
#         if score <= 2:
#             return 'Negatif'
#         elif score == 3:
#             return 'Netral'
#         else:
#             return 'Positif'

#     # Terapkan fungsi get_sentiment ke kolom 'score'
#     df['label'] = df['score'].apply(get_sentiment)

#     return df

In [99]:
# print("Labeling data berdasarkan skor bintang")
# df_labeled = labeling_data(df)

In [100]:
def label_with_indobert(df):
  pretrained_name = "w11wo/indonesian-roberta-base-sentiment-classifier"

  nlp = pipeline(
      "sentiment-analysis",
      model=pretrained_name,
      tokenizer=pretrained_name,
      truncation=True, # Potong teks jika terlalu panjang (>512 kata)
      max_length=512
    )

  labels = []
  scores = []

  for text in tqdm(df['soft_clean_content']):
    try:
      result = nlp(text)[0] # Prediksi sentimen

      label = result['label'] # output: 'positive', 'neutral', 'negative'

      # Mapping ulang ke format Bahasa Indonesia
      label_map = {
        'positive': 'Positif',
        'neutral': 'Netral',
        'negative': 'Negatif'
      }
      labels.append(label_map.get(label, label))
      scores.append(result['score'])

    except Exception as e:
      print(f"Error pada teks: {text}")
      labels.append("Netral")
      scores.append(0.0)

  df['sentiment_label'] = labels
  df['confidence_score'] = scores

  return df

In [ ]:
df_labeled = label_with_indobert(df)

In [102]:
df_labeled.head(5)

,content,score,soft_clean_content,sentiment_label,confidence_score
0,"suka belanja di shopee,jaya selalu shopee",5,"suka belanja di shopee,jaya selalu shopee",Positif,0.980013
1,Banyak ongkir gratis nya gak akan rugi beli di...,5,Banyak ongkir gratis nya gak akan rugi beli di...,Positif,0.998416
2,saya sangat puas harga nya murah murah,5,saya sangat puas harga nya murah murah,Positif,0.998704
3,belanja jadi mudah dan menyenangkan tak perlu ...,5,belanja jadi mudah dan menyenangkan tak perlu ...,Positif,0.995944
4,kurirnya ga jelas kirim pesan tidak menghubung...,1,kurirnya ga jelas kirim pesan tidak menghubung...,Negatif,0.996652


**Pemeriksaan Distribusi Data, memastikan apakah terdapat imbalance atau tidak**

In [103]:
print("Distribusi Data per Kelas:")
print(df_labeled['sentiment_label'].value_counts())

Distribusi Data per Kelas:
sentiment_label
Positif    38388
Negatif    31806
Netral      4990
Name: count, dtype: int64


**Konversi ke csv untuk mempermudah dalam pembersihan data lebih lanjut**

In [104]:
nama_file = 'dataset_ulasan_playstore.csv'
df_labeled.to_csv(nama_file, index=False)

print(f"Selesai! Data berhasil disimpan ke '{nama_file}'")
print(f"Total data: {len(df_labeled)}")

Selesai! Data berhasil disimpan ke 'dataset_ulasan_playstore.csv'
Total data: 75184
